In [1]:
import sys
import os

from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as ss

In [2]:
from kmer_model import KmerLinearRegressor

In [3]:
src_file = "../../predictor/regression_multiple/UTR3_zscores_replicateagg.csv"
src_file_code = os.path.basename(src_file).partition("_")[0]
data = pd.read_csv(src_file)
data

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c1,train,6.809279,5.843161,5.836944,23.948600,3.105728,2.860766,0.244962,1.165330,0.210208
1,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c13,train,42.009822,34.532234,32.334772,56.966098,2.628650,2.860766,-0.232116,-1.104220,0.210208
2,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c17,train,50.033644,45.049730,54.291393,110.129539,2.865176,2.860766,0.004410,0.020978,0.210208
3,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c2,train,26.001559,44.713552,22.349163,57.504485,2.739573,2.860766,-0.121193,-0.576539,0.210208
4,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c4,train,12.116972,26.338286,33.479228,67.473281,3.121235,2.860766,0.260469,1.239099,0.210208
...,...,...,...,...,...,...,...,...,...,...,...,...
170551,TTTTTTTTTTTTTTTTTTTTTTTTGGTGTGTGTGTGTGTGTGTGTG...,c13,train,20.230416,15.444105,17.989054,34.789154,2.761276,2.669052,0.092224,0.517556,0.178191
170552,TTTTTTTTTTTTTTTTTTTTTTTTGGTGTGTGTGTGTGTGTGTGTG...,c17,train,50.443141,46.962936,78.430440,73.845290,2.703607,2.669052,0.034555,0.193919,0.178191
170553,TTTTTTTTTTTTTTTTTTTTTTTTGGTGTGTGTGTGTGTGTGTGTG...,c2,train,25.691550,24.562474,16.716270,22.217946,2.397593,2.669052,-0.271460,-1.523421,0.178191
170554,TTTTTTTTTTTTTTTTTTTTTTTTGGTGTGTGTGTGTGTGTGTGTG...,c4,train,12.025867,17.908212,22.000636,31.344779,2.872536,2.669052,0.203483,1.141940,0.178191


In [4]:
data_unique = data[["seq", "cell_type", "fold", "mass_center"]].drop_duplicates()
train = data_unique[data_unique["fold"] == "train"]
test = data_unique[data_unique["fold"] == "test"]

In [5]:
train_averaged = train.groupby("seq")["mass_center"].mean().reset_index()
test_averaged = test.groupby("seq")["mass_center"].mean().reset_index()

## Defining cell types

In [6]:
cell_types = sorted(data["cell_type"].unique())
cell_types

['c1', 'c13', 'c17', 'c2', 'c4', 'c6']

## Creating regression models

In [7]:
pred_results = {}
metrics_results = {}

for kmer_length in (1, 2, 3):
    metrics_results[kmer_length] = metrics = defaultdict(dict)
    regressor = KmerLinearRegressor(complement=False, kmer_length=kmer_length,
                                    linreg_kws=dict(random_state=0))
    regressor.fit(train_averaged["seq"], train_averaged["mass_center"])
    test_averaged["pred_mass_center"] = regressor.predict(test_averaged["seq"])
    metrics["r"]["all_avg"] = ss.pearsonr(test_averaged["mass_center"], test_averaged["pred_mass_center"]).statistic
    metrics["rho"]["all_avg"] = ss.spearmanr(test_averaged["mass_center"], test_averaged["pred_mass_center"]).statistic

    test_copy = test.copy()
    test_copy["pred_mass_center"] = -1.0
    for ct, ctdf in train.groupby(by="cell_type"):
        regressor = KmerLinearRegressor(complement=False, kmer_length=kmer_length,
                                        linreg_kws=dict(random_state=0))
        regressor.fit(ctdf["seq"], ctdf["mass_center"])
        pred = regressor.predict(test_copy[test_copy["cell_type"] == ct]["seq"])
        test_copy.loc[test_copy["cell_type"] == ct, "pred_mass_center"] = pred

    pred_results[("pred_mass_center", f"k={kmer_length}")] = test_copy["pred_mass_center"]

    g = test_copy.groupby("seq")
    metrics["r"]["all"] = ss.pearsonr(g["mass_center"].mean(), g["pred_mass_center"].mean()).statistic
    metrics["rho"]["all"] = ss.spearmanr(g["mass_center"].mean(), g["pred_mass_center"].mean()).statistic
    for ct, ctdf in test_copy.groupby(by="cell_type"):
        metrics["r"][ct] = ss.pearsonr(ctdf["mass_center"], ctdf["pred_mass_center"]).statistic
        metrics["rho"][ct] = ss.spearmanr(ctdf["mass_center"], ctdf["pred_mass_center"]).statistic

In [8]:
pred_results_df = pd.DataFrame(pred_results).sort_index(axis=1)
pred_results_df.insert(0, "seq", test["seq"])
pred_results_df.insert(1, "cell_type", test["cell_type"])
pred_results_df.insert(2, "mass_center", test["mass_center"])
pred_results_df

seq cell_type  \
                                                                      
180     AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...        c1   
181     AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...       c13   
182     AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...       c17   
183     AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...        c2   
184     AAAAAAAGAGAGACCTGTCATTAGAAGCAACCAGGTTCTCCTGATA...        c4   
...                                                   ...       ...   
170485  TTTTTTTTTTTTTTTCCGTCTCCCCAAAGCTTTATCTGTCTTGACT...       c13   
170486  TTTTTTTTTTTTTTTCCGTCTCCCCAAAGCTTTATCTGTCTTGACT...       c17   
170487  TTTTTTTTTTTTTTTCCGTCTCCCCAAAGCTTTATCTGTCTTGACT...        c2   
170488  TTTTTTTTTTTTTTTCCGTCTCCCCAAAGCTTTATCTGTCTTGACT...        c4   
170489  TTTTTTTTTTTTTTTCCGTCTCCCCAAAGCTTTATCTGTCTTGACT...        c6   

       mass_center pred_mass_center                      
                                k=1       k=2       k=3  
180       2.433101         2.379173  2.425182  2.473915  
181       2.427481         2.478421  2.520384  2.564177  
182       2.897645         2.478960  2.594341  2.714377  
183       2.591041         2.423796  2.487797  2.549767  
184       2.573348         2.438466  2.539621  2.630256  
...            ...              ...       ...       ...  
170485    2.722813         2.574745  2.625760  2.570931  
170486    2.787094         2.662443  2.751325  2.705869  
170487    2.520863         2.682263  2.737788  2.663821  
170488    2.891843         2.619400  2.709630  2.665772  
170489    2.881207         2.653249  2.706985  2.638353  

[17052 rows x 6 columns]

In [9]:
pred_results_df.to_parquet("models_kmer_preds_utr3.parquet")

In [10]:
import json

with open("models_kmer_utr3.json", "wt") as outfile:
    json.dump(metrics_results, outfile)